# Run PreciseWikiQA Inference via Python Function

This notebook runs the equivalent of:

`scripts/run_with_server.py --step inference --task precisewikiqa --model meta-llama/Llama-3.1-8B-Instruct --N 100 --logger-type zarr --activations-path shared/goodwiki.zarr/activations.zarr --log-file shared/goodwiki.zarr/server.log`

but through Python by calling `run_experiment(...)` from `scripts/run_with_server.py`.

In [ ]:
from pathlib import Path
import sys

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "scripts" / "run_with_server.py").exists():
    raise RuntimeError("Open this notebook from the HalluLens repository root.")

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from scripts.run_with_server import run_experiment

print(f"Repo root: {REPO_ROOT}")

In [ ]:
model = "meta-llama/Llama-3.1-8B-Instruct"
task = "precisewikiqa"
step = "inference"
N = 1000

wiki_src = "goodwiki"
mode = "dynamic"
inference_method = "vllm"

logger_type = "lmdb"
activations_path = "shared/precisewikilogprob/activations.zarr"
log_file = "shared/precisewikilogprob/server.log"

host = "0.0.0.0"
port = 8000

Path(activations_path).parent.mkdir(parents=True, exist_ok=True)
print(f"Activation directory ready: {Path(activations_path).parent}")

qa_output_path = f"data/precise_qa/save/qa_{wiki_src}_{model.split('/')[-1]}_{mode}.jsonl"
if not Path(qa_output_path).exists():
    raise FileNotFoundError(
        f"Missing QA file: {qa_output_path}\n"
        "Run generation first (next optional cell) or use an existing qa_output_path."
    )

print(f"Found QA file: {qa_output_path}")

run_experiment(
    step=step,
    task=task,
    model=model,
    N=N,
    wiki_src=wiki_src,
    mode=mode,
    inference_method=inference_method,
    logger_type=logger_type,
    activations_path=activations_path,
    log_file=log_file,
    host=host,
    port=port,
)

In [ ]:
import hashlib
import json
from pathlib import Path

import numpy as np

from activation_logging.activations_logger import ActivationsLogger

store_path = Path(activations_path)
if not store_path.exists():
    raise FileNotFoundError(f"Activation store not found: {store_path}")

task_name = f"precise_wikiqa_{wiki_src}_{mode}"
model_name = model.split("/")[-1]
gen_path = Path("output") / task_name / model_name / "generation.jsonl"
if "generations_file_path" in globals() and generations_file_path:
    gen_path = Path(generations_file_path)

reader = ActivationsLogger(lmdb_path=str(store_path), read_only=True, verbose=False)
try:
    all_keys = reader.list_entries()
    if not all_keys:
        raise RuntimeError(
            f"No activation entries found in {store_path}. "
            "Inference may not have logged activations."
        )

    max_entries_to_check = min(300, len(all_keys))
    checked_keys = all_keys[:max_entries_to_check]

    entries_with_logprobs = 0
    total_response_tokens = 0
    finite_token_logprobs = 0
    finite_topk_logprobs = 0
    total_topk_values = 0
    positive_token_logprobs = 0
    monotonicity_violations = 0
    selected_not_in_topk = 0
    selected_not_top1 = 0

    token_logprobs_values = []
    top1_logprobs_values = []
    first_payload_example = None

    for key in checked_keys:
        payload = reader.get_response_logprobs(key)
        if payload is None:
            continue

        entries_with_logprobs += 1
        token_ids = np.asarray(payload["response_token_ids"])
        token_logprobs = np.asarray(payload["response_token_logprobs"], dtype=np.float32)
        topk_ids = np.asarray(payload["response_topk_token_ids"])
        topk_logprobs = np.asarray(payload["response_topk_logprobs"], dtype=np.float32)

        if token_logprobs.ndim != 1 or topk_logprobs.ndim != 2:
            raise AssertionError(
                f"Malformed shapes for key={key}: token_logprobs={token_logprobs.shape}, "
                f"topk_logprobs={topk_logprobs.shape}"
            )

        usable_len = min(
            token_ids.shape[0],
            token_logprobs.shape[0],
            topk_ids.shape[0],
            topk_logprobs.shape[0],
        )
        if usable_len <= 0:
            continue

        token_ids = token_ids[:usable_len]
        token_logprobs = token_logprobs[:usable_len]
        topk_ids = topk_ids[:usable_len]
        topk_logprobs = topk_logprobs[:usable_len]

        total_response_tokens += usable_len
        finite_token_logprobs += int(np.isfinite(token_logprobs).sum())
        finite_topk_logprobs += int(np.isfinite(topk_logprobs).sum())
        total_topk_values += int(topk_logprobs.size)
        positive_token_logprobs += int((token_logprobs > 1e-6).sum())

        if topk_logprobs.shape[1] > 1:
            monotonicity_violations += int((np.diff(topk_logprobs, axis=1) > 1e-6).sum())

        selected_not_top1 += int((token_ids != topk_ids[:, 0]).sum())
        selected_not_in_topk += int((token_logprobs < (topk_logprobs[:, -1] - 1e-6)).sum())

        token_logprobs_values.append(token_logprobs)
        top1_logprobs_values.append(topk_logprobs[:, 0])

        if first_payload_example is None:
            first_payload_example = {
                "key": key,
                "token_ids": token_ids,
                "token_logprobs": token_logprobs,
                "topk_ids": topk_ids,
                "topk_logprobs": topk_logprobs,
            }

    if entries_with_logprobs == 0 or total_response_tokens == 0:
        raise RuntimeError(
            "No response logprobs were found in checked activation entries. "
            "Confirm logprob capture is enabled on the inference server."
        )

    token_logprobs_all = np.concatenate(token_logprobs_values)
    top1_logprobs_all = np.concatenate(top1_logprobs_values)

    token_finite_ratio = finite_token_logprobs / max(1, total_response_tokens)
    topk_finite_ratio = finite_topk_logprobs / max(1, total_topk_values)
    selected_not_top1_ratio = selected_not_top1 / max(1, total_response_tokens)

    issues = []
    if token_finite_ratio < 0.999:
        issues.append(f"Only {token_finite_ratio:.4f} of selected-token logprobs are finite")
    if topk_finite_ratio < 0.999:
        issues.append(f"Only {topk_finite_ratio:.4f} of top-k logprobs are finite")
    if positive_token_logprobs > 0:
        issues.append(f"Found {positive_token_logprobs} selected-token logprobs > 0")
    if monotonicity_violations > 0:
        issues.append(f"Found {monotonicity_violations} top-k monotonicity violations")
    if selected_not_in_topk > 0:
        issues.append(
            f"Found {selected_not_in_topk} tokens where selected logprob is below stored top-k minimum"
        )

    print("Logprob sanity check summary")
    print(f"- Activation store: {store_path}")
    print(f"- Entries in store: {len(all_keys)}")
    print(f"- Entries checked: {max_entries_to_check}")
    print(f"- Entries with logprobs: {entries_with_logprobs}")
    print(f"- Response tokens checked: {total_response_tokens}")
    print(f"- Selected-token finite ratio: {token_finite_ratio:.4f}")
    print(f"- Top-k finite ratio: {topk_finite_ratio:.4f}")
    print(
        "- Selected-token logprob quantiles (p01/p50/p99): "
        f"{np.quantile(token_logprobs_all, 0.01):.3f} / "
        f"{np.quantile(token_logprobs_all, 0.50):.3f} / "
        f"{np.quantile(token_logprobs_all, 0.99):.3f}"
    )
    print(
        "- Top-1 logprob quantiles (p01/p50/p99): "
        f"{np.quantile(top1_logprobs_all, 0.01):.3f} / "
        f"{np.quantile(top1_logprobs_all, 0.50):.3f} / "
        f"{np.quantile(top1_logprobs_all, 0.99):.3f}"
    )
    print(f"- Selected token != top-1 candidate ratio: {selected_not_top1_ratio:.4f}")

    if gen_path.exists() and first_payload_example is not None:
        sampled_key = first_payload_example["key"]
        sampled_hash = sampled_key.split("_", 1)[0]
        matched_record = None

        with open(gen_path, "r", encoding="utf-8") as f:
            for line in f:
                line = line.strip()
                if not line:
                    continue
                rec = json.loads(line)
                prompt_text = rec.get("prompt")
                if not isinstance(prompt_text, str):
                    continue
                if hashlib.sha256(prompt_text.encode("utf-8")).hexdigest() == sampled_hash:
                    matched_record = rec
                    break

        if matched_record is not None:
            prompt_preview = matched_record["prompt"].replace("\n", " ")[:160]
            generation_preview = str(matched_record.get("generation", "")).replace("\n", " ")[:160]
            print("- Example matched generation row:")
            print(f"  key={sampled_key}")
            print(f"  prompt={prompt_preview}...")
            print(f"  generation={generation_preview}...")
            print(
                "  first 5 selected token logprobs:",
                np.round(first_payload_example["token_logprobs"][:5], 3).tolist(),
            )
        else:
            print(f"- Could not match sampled key in generations file: {gen_path}")
    elif gen_path.exists():
        print(f"- Generations file found: {gen_path} (no logprob payload sample available)")
    else:
        print(f"- Generations file not found at {gen_path}; skipped prompt/output cross-check.")

    if issues:
        print("\nPotential issues detected:")
        for issue in issues:
            print(f"- {issue}")
        raise AssertionError("Logprob sanity check failed.")

    print("\nPASS: Response logprobs are present and look numerically normal.")
finally:
    reader.close()

In [ ]:
# model = "meta-llama/Llama-3.1-8B-Instruct"
# task = "naturalquestions"
# step = "inference"
# N = 1000

# inference_method = "vllm"
# max_tokens = 64
# temperature = 0.0

# logger_type = "zarr"
# activations_path = "shared/natural_questions_dev/activations.zarr"
# log_file = "shared/natural_questions_dev/server.log"

# data_dir = "external/LLMsKnow/data"
# activations_dir = Path(activations_path).parent
# output_dir = str(activations_dir) if str(activations_dir).strip() not in {"", "."} else "output"

# host = "0.0.0.0"
# port = 8000

# activations_dir.mkdir(parents=True, exist_ok=True)
# print(f"Activation directory ready: {activations_dir}")
# print(f"Output directory: {output_dir}")

# run_experiment(
#     step=step,
#     task=task,
#     model=model,
#     inference_method=inference_method,
#     N=N,
#     max_tokens=max_tokens,
#     temperature=temperature,
#     logger_type=logger_type,
#     activations_path=activations_path,
#     log_file=log_file,
#     data_dir=data_dir,
#     output_dir=output_dir,
#     host=host,
#     port=port,
# )